In [1]:
!pip install -q "scikit-learn==1.6.1" joblib requests

In [2]:
import io
import json
import zipfile

import requests

DATASET_URL = (
    "https://cdn.services-k8s.prod.aws.htb.systems/content/modules/292/"
    "skills_assessment_data.zip"
)

print("[i] Downloading dataset...")
resp = requests.get(DATASET_URL, timeout=120)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    with zf.open("train.json") as f:
        train_data = json.load(f)
    with zf.open("test.json") as f:
        test_data = json.load(f)

print(f"[i] Train samples: {len(train_data)}")
print(f"[i] Test  samples: {len(test_data)}")
print("[i] Example:", {k: (v[:80] + "..." if isinstance(v, str) and len(v) > 80 else v)
                      for k, v in train_data[0].items()})

[i] Downloading dataset...
[i] Train samples: 25000
[i] Test  samples: 25000
[i] Example: {'text': 'Bromwell High is a cartoon comedy. It ran at the same time as some other program...', 'label': 1}


In [3]:
X_train = [s["text"] for s in train_data]
y_train = [s["label"] for s in train_data]
X_test = [s["text"] for s in test_data]
y_test = [s["label"] for s in test_data]

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Label balance (train): 0={y_train.count(0)}, 1={y_train.count(1)}")

Train size: 25000 | Test size: 25000
Label balance (train): 0=12500, 1=12500


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import joblib

pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

print("[i] Training...")
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"[i] Test accuracy: {acc * 100:.2f}%")
print(classification_report(y_test, y_pred, digits=4))

[i] Training...
[i] Test accuracy: 88.29%
              precision    recall  f1-score   support

           0     0.8818    0.8844    0.8831     12500
           1     0.8841    0.8814    0.8827     12500

    accuracy                         0.8829     25000
   macro avg     0.8829    0.8829    0.8829     25000
weighted avg     0.8829    0.8829    0.8829     25000



In [5]:
MODEL_PATH = "skills_assessment.joblib"
joblib.dump(pipeline, MODEL_PATH)
print(f"[i] Saved -> {MODEL_PATH}")

# Quick sanity check
samples = [
    "This movie was fantastic, brilliant acting and a wonderful story.",
    "Terrible film, waste of time, boring and poorly made.",
]
print("[i] Sample predictions:", pipeline.predict(samples), "(expect [1, 0])")

[i] Saved -> skills_assessment.joblib
[i] Sample predictions: [1 0] (expect [1, 0])


In [3]:
import requests
import json

# Define the URL of the API endpoint
url = "http://10.129.98.119:5000/api/upload"

# Path to the model file you want to upload
model_file_path = "skills_assessment.joblib"

# Open the file in binary mode and send the POST request
with open(model_file_path, "rb") as model_file:
    files = {"model": model_file}
    response = requests.post(url, files=files)

# Pretty print the response from the server
print(json.dumps(response.json(), indent=4))

{
    "accuracy": 1.0,
    "flag": "HTB{s3nt1m3nt_4n4lys1s_d4t4}"
}
